# LeetCode 753 — Cracking the Safe bằng Python

- Link: https://leetcode.com/problems/cracking-the-safe/
- Độ khó: **Hard**
- Chủ đề: Đồ thị de Bruijn, Chu trình Euler (thuật toán Hierholzer), DFS/Backtracking, Tham lam, Chuỗi de Bruijn

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def crackSafe(self, n: int, k: int) -> str:
```

## 1. Giới thiệu & mục tiêu học

Một chiếc két sắt có mật khẩu gồm `n` chữ số, mỗi chữ số thuộc `0..k-1`. Két **chỉ nhớ `n` chữ số gõ gần nhất**: mỗi lần gõ thêm một phím, nó so `n` phím cuối với mật khẩu. Ta muốn gõ **một chuỗi ngắn nhất** sao cho dù mật khẩu là gì, tại một thời điểm nào đó két cũng mở.

Nghe như bài chuỗi, nhưng thực chất là một bài **đồ thị kinh điển**: tìm **chu trình Euler** trên **đồ thị de Bruijn**. Đây là ví dụ đẹp nhất để học Hierholzer.

Sau notebook này bạn sẽ:

1. Hiểu vì sao độ dài tối thiểu là đúng `k^n + n - 1`.
2. Viết lời giải **backtracking** (đi qua mọi mật khẩu đúng một lần) — baseline dễ hiểu.
3. Mô hình hoá bài toán thành **đồ thị de Bruijn** và tìm **chu trình Euler** bằng **Hierholzer** (đệ quy và lặp).
4. Biết lời giải **tham lam "chữ số lớn nhất trước"** và cách dựng **chuỗi de Bruijn bằng từ Lyndon** (thuật toán FKM).
5. Viết bộ kiểm tra tính hợp lệ của đáp án (vì bài chấp nhận **nhiều** đáp án).

## 2. Đề bài & ví dụ

**Phát biểu lại.** Mật khẩu là một chuỗi độ dài `n`, mỗi ký tự là chữ số trong `[0, k-1]`. Khi ta gõ một dãy phím, sau mỗi phím két kiểm tra **`n` phím gần nhất**. Hãy trả về **một chuỗi bất kỳ có độ dài nhỏ nhất** mà chắc chắn mở được két ở một thời điểm nào đó, tức là **mọi** chuỗi độ dài `n` trên bảng chữ `{0..k-1}` đều xuất hiện như một **chuỗi con liên tiếp** của đáp án.

**Ví dụ 1**

```text
Input: n = 1, k = 2
Output: "10"
Giải thích: "01" cũng đúng. Mật khẩu chỉ có thể là "0" hoặc "1", cả hai đều xuất hiện.
```

**Ví dụ 2**

```text
Input: n = 2, k = 2
Output: "01100"
Giải thích: các chuỗi con độ dài 2 là "01", "11", "10", "00" — đủ cả 4 mật khẩu.
"00110", "10011", "11001" cũng được chấp nhận.
```

**Ràng buộc**

- `1 <= n <= 4`
- `1 <= k <= 10`
- `1 <= k^n <= 4096`

Vì có nhiều đáp án, ta **không so sánh với một chuỗi cố định** mà viết hàm `is_valid` kiểm tra: (1) độ dài đúng `k^n + n - 1`, (2) mọi ký tự trong `0..k-1`, (3) số chuỗi con độ dài `n` **khác nhau** đúng bằng `k^n`.

In [1]:
def is_valid(s, n, k):
    """Kiểm tra s có phải đáp án tối ưu của Cracking the Safe hay không."""
    if len(s) != k ** n + n - 1:
        return False
    if any(not ('0' <= c <= str(k - 1)) for c in s):
        return False
    windows = {s[i:i + n] for i in range(len(s) - n + 1)}
    return len(windows) == k ** n

print(is_valid("10", 1, 2), is_valid("01", 1, 2))
print(is_valid("01100", 2, 2), is_valid("00110", 2, 2))
print(is_valid("0011", 2, 2))   # thiếu "10" và độ dài sai

True True
True True
False


## 3. Kiến thức nền

### 3.1. Đếm số chuỗi độ dài `n` trên bảng chữ `k` ký tự

Mỗi vị trí có `k` lựa chọn, độc lập nhau → `k · k · ... · k = k^n` chuỗi. Ta có thể sinh hết bằng `itertools.product`.

In [2]:
from itertools import product

n, k = 2, 3
all_pw = [''.join(p) for p in product('012'[:k], repeat=n)]
print(len(all_pw), all_pw)

9 ['00', '01', '02', '10', '11', '12', '20', '21', '22']


### 3.2. Cửa sổ trượt trên chuỗi và slicing

Chuỗi `s` có `len(s) - n + 1` cửa sổ độ dài `n`: `s[0:n], s[1:n+1], ...`. Khi đi từ cửa sổ này sang cửa sổ kế tiếp, ta **bỏ ký tự đầu** và **thêm một ký tự cuối**: `nxt = cur[1:] + c`. Đây là phép biến đổi trung tâm của bài.

In [3]:
s = "01100"
n = 2
print([s[i:i + n] for i in range(len(s) - n + 1)])

cur = "01"
for c in "100":
    cur = cur[1:] + c
    print("thêm", c, "->", cur)

['01', '11', '10', '00']
thêm 1 -> 11
thêm 0 -> 10
thêm 0 -> 00


### 3.3. Đồ thị có hướng, bậc vào/bậc ra và chu trình Euler

- **Đồ thị có hướng**: cạnh `u → v` có chiều.
- **Bậc ra** `out(u)`: số cạnh đi ra khỏi `u`; **bậc vào** `in(u)`: số cạnh đi vào `u`.
- **Đường đi Euler**: đi qua **mọi cạnh đúng một lần**. **Chu trình Euler**: đường đi Euler quay về điểm xuất phát.
- **Định lý**: đồ thị có hướng (các đỉnh có cạnh đều liên thông) có chu trình Euler **khi và chỉ khi** mọi đỉnh có `in(u) = out(u)`.

So sánh: **đường đi Hamilton** đi qua **mọi đỉnh** đúng một lần — bài toán NP-khó nói chung. Mẹo của bài này là đổi bài toán Hamilton (khó) sang bài toán Euler (dễ).

In [4]:
from collections import defaultdict

edges = [("A", "B"), ("B", "C"), ("C", "A"), ("A", "C"), ("C", "A")]
indeg, outdeg = defaultdict(int), defaultdict(int)
for u, v in edges:
    outdeg[u] += 1
    indeg[v] += 1
for x in "ABC":
    print(x, "in =", indeg[x], "out =", outdeg[x])
# Mọi đỉnh in == out -> có chu trình Euler

A in = 2 out = 2
B in = 1 out = 1
C in = 2 out = 2


### 3.4. Thuật toán Hierholzer (ý tưởng)

1. Bắt đầu từ một đỉnh, **đi theo cạnh chưa dùng** cho đến khi kẹt (khi kẹt, chắc chắn ta đang ở đỉnh xuất phát nếu mọi đỉnh cân bằng).
2. Khi một đỉnh **hết cạnh**, ghi nó vào kết quả (**post-order**) và lùi lại.
3. Kết quả đảo ngược chính là chu trình Euler. Các "vòng phụ" được tự động ghép vào vòng chính nhờ thứ tự post-order.

Dưới đây là Hierholzer tổng quát trên đồ thị nhỏ ở 3.3.

In [5]:
def hierholzer(adj, start):
    adj = {u: list(vs) for u, vs in adj.items()}   # bản sao để xoá cạnh
    stack, circuit = [start], []
    while stack:
        u = stack[-1]
        if adj.get(u):
            stack.append(adj[u].pop())   # đi theo một cạnh chưa dùng
        else:
            circuit.append(stack.pop())  # hết cạnh: ghi post-order
    return circuit[::-1]

adj = defaultdict(list)
for u, v in edges:
    adj[u].append(v)
print(hierholzer(adj, "A"))   # 6 đỉnh = 5 cạnh + 1

['A', 'C', 'A', 'B', 'C', 'A']


### 3.5. Đệ quy sâu trong Python

Đệ quy có giới hạn mặc định khoảng 1000 tầng (`sys.getrecursionlimit()`). Ở bài này độ sâu có thể tới `k^n = 4096`, nên khi viết đệ quy ta phải tăng giới hạn bằng `sys.setrecursionlimit`, hoặc **viết bản lặp với stack tự quản** (an toàn hơn).

In [6]:
import sys
print("giới hạn hiện tại:", sys.getrecursionlimit())
sys.setrecursionlimit(20000)

def depth(d):
    return 0 if d == 0 else 1 + depth(d - 1)

print(depth(5000))

giới hạn hiện tại: 1000
5000


## 4. Ý tưởng nền tảng / trực giác

### 4.1. Cận dưới của độ dài

Có `k^n` mật khẩu cần "phủ". Chuỗi độ dài `L` có `L - n + 1` cửa sổ độ dài `n`. Muốn có đủ `k^n` cửa sổ khác nhau thì cần `L - n + 1 >= k^n`, tức

$$L \ge k^n + n - 1.$$

Nếu đạt đúng dấu bằng, **mọi cửa sổ phải khác nhau**: mỗi mật khẩu xuất hiện **đúng một lần**. Điều kỳ diệu là luôn đạt được (chuỗi de Bruijn).

### 4.2. Tính tay với n = 2, k = 2

Mật khẩu: `00, 01, 10, 11`. Cần chuỗi dài `4 + 2 - 1 = 5`.

- Bắt đầu `00`. Thêm `1` → cửa sổ mới `01`. Chuỗi `001`.
- Từ `01` thêm `1` → `11`. Chuỗi `0011`.
- Từ `11` thêm `0` → `10`. Chuỗi `00110`. Đủ 4 cửa sổ: `00, 01, 11, 10`.

### 4.3. Tính tay với n = 2, k = 3 (9 mật khẩu, độ dài 10)

Bắt đầu `00`, mỗi bước chọn chữ số sao cho cửa sổ mới chưa gặp:

| bước | chuỗi | cửa sổ mới |
|---|---|---|
| 0 | `00` | `00` |
| 1 | `001` | `01` |
| 2 | `0011` | `11` |
| 3 | `00110` | `10` |
| 4 | `001102` | `02` |
| 5 | `0011021` | `21` |
| 6 | `00110212` | `12` |
| 7 | `001102122` | `22` |
| 8 | `0011021220` | `20` |

Chuỗi `0011021220` dài 10, đủ 9 cửa sổ. Nhưng nếu chọn ẩu (ví dụ từ `00` đi `01`, `10`, rồi lại về `00`? không được vì `00` đã dùng) có thể bị **kẹt** trước khi đủ — đó là lý do cần backtracking hoặc một thuật toán thông minh hơn.

In [7]:
s = "0011021220"
print(is_valid(s, 2, 3), len(s))

True 10


## 5. Cách 1 — Brute Force: Backtracking qua các mật khẩu (đường đi Hamilton)

**Ý tưởng.** Coi mỗi mật khẩu (chuỗi độ dài `n`) là một **đỉnh**. Từ đỉnh `u` có cạnh sang `u[1:] + c` với mỗi `c`. Ta cần đường đi qua **mọi đỉnh đúng một lần** — đường Hamilton. Dùng DFS + quay lui: bắt đầu `'0'*n`, thử từng chữ số, nếu cửa sổ mới chưa thăm thì đi tiếp; nếu kẹt mà chưa đủ thì quay lui.

Với đồ thị de Bruijn, DFS này **gần như không bao giờ phải quay lui** (chạy rất nhanh trong thực tế), nhưng về lý thuyết backtracking có thể bùng nổ mũ, và ta chưa có lý do chặt chẽ vì sao nó luôn tìm được.

In [8]:
import sys
sys.setrecursionlimit(20000)

def crack_backtrack(n, k):
    total = k ** n
    start = '0' * n
    seen = {start}
    path = [start]
    stats = {"backtrack": 0}

    def dfs(node):
        if len(seen) == total:
            return True
        for x in map(str, range(k)):
            nei = node[1:] + x
            if nei not in seen:
                seen.add(nei)
                path.append(x)
                if dfs(nei):
                    return True
                seen.remove(nei)       # quay lui
                path.pop()
                stats["backtrack"] += 1
        return False

    dfs(start)
    return ''.join(path), stats["backtrack"]

for n, k in [(1, 2), (2, 2), (2, 3), (3, 2), (4, 10)]:
    s, bt = crack_backtrack(n, k)
    print(n, k, "len =", len(s), "valid =", is_valid(s, n, k), "số lần quay lui =", bt)

1 2 len = 2 valid = True số lần quay lui = 0
2 2 len = 5 valid = True số lần quay lui = 1
2 3 len = 10 valid = True số lần quay lui = 2
3 2 len = 10 valid = True số lần quay lui = 3
4 10 len = 10003 valid = True số lần quay lui = 54


**Độ phức tạp.** Mỗi bước thử `k` chữ số, tạo chuỗi dài `n` → `O(n·k)` mỗi đỉnh. Nếu không quay lui: `O(n · k · k^n)`. Trường hợp xấu nhất về lý thuyết (quay lui) là mũ theo số đỉnh. Bộ nhớ `O(n · k^n)` cho tập `seen` + độ sâu đệ quy `k^n`.

**Nút thắt.**

1. Ta đang giải bài **Hamilton** — không có đảm bảo đa thức nói chung.
2. Đệ quy sâu `k^n = 4096` tầng — phải tăng recursion limit.

→ Cần một góc nhìn để biến bài thành bài **Euler**, có thuật toán tuyến tính và có chứng minh.

## 6. Bước đột phá

**Insight: đổi vai trò đỉnh và cạnh.**

- Cho **đỉnh** là các chuỗi độ dài **`n - 1`** (có `k^(n-1)` đỉnh).
- Mỗi **mật khẩu** `p` (độ dài `n`) là một **cạnh** từ `p[:-1]` đến `p[1:]`. Có đúng `k^n` cạnh.
- Đi theo cạnh `p` từ đỉnh `u = p[:-1]` nghĩa là **gõ thêm chữ số `p[-1]`**; ta đến đỉnh `v = u[1:] + p[-1]`.

Khi đó "gõ một chuỗi chứa mọi mật khẩu đúng một lần" ⇔ "đi qua **mọi cạnh đúng một lần**" ⇔ **đường đi Euler**!

**Kiểm tra điều kiện Euler.** Mỗi đỉnh `u` (độ dài `n-1`):

- **Bậc ra = k**: thêm một chữ số bất kỳ `c` vào cuối → cạnh `u + c`.
- **Bậc vào = k**: thêm một chữ số bất kỳ `c` vào đầu → cạnh `c + u`.
- Liên thông mạnh: từ `u` gõ `n-1` chữ số của `v` là tới `v`.

Vậy luôn tồn tại **chu trình Euler** → luôn đạt độ dài tối thiểu `k^n + n - 1`: gõ `n-1` chữ số của đỉnh xuất phát, rồi `k^n` chữ số, mỗi chữ số ứng với một cạnh.

**Quy tắc nếu… thì…**

- Nếu bài yêu cầu "chuỗi ngắn nhất chứa mọi chuỗi con độ dài `n`" → nghĩ ngay **de Bruijn + Euler**.
- Nếu đang cần "đi qua mọi đỉnh" mà mỗi đỉnh là chuỗi chồng lấn → thử **hạ một bậc**: đỉnh = chuỗi ngắn hơn 1, cạnh = chuỗi dài.
- Nếu `n = 1` → đỉnh là chuỗi rỗng, `k` khuyên (self-loop): đáp án chỉ là `"01...(k-1)"`.

### 6.1. Minh hoạ đồ thị de Bruijn với n = 3, k = 2

Đỉnh (độ dài 2): `00, 01, 10, 11`. Cạnh (độ dài 3), ghi dạng `u --c--> v`:

```text
00 --0--> 00   (cạnh 000)     00 --1--> 01   (cạnh 001)
01 --0--> 10   (cạnh 010)     01 --1--> 11   (cạnh 011)
10 --0--> 00   (cạnh 100)     10 --1--> 01   (cạnh 101)
11 --0--> 10   (cạnh 110)     11 --1--> 11   (cạnh 111)
```

Mỗi đỉnh có 2 cạnh ra, 2 cạnh vào. Một chu trình Euler từ `00`: `000 → 001 → 011 → 111 → 110 → 101 → 010 → 100`, gõ: `00` + `0 1 1 1 0 1 0 0` = `0001110100` (dài 10 = 8 + 3 - 1).

In [9]:
def de_bruijn_graph(n, k):
    nodes = [''.join(p) for p in product(map(str, range(k)), repeat=n - 1)]
    return {u: [(u + c, (u + c)[1:]) for c in map(str, range(k))] for u in nodes}

g = de_bruijn_graph(3, 2)
for u, outs in g.items():
    print(u, "->", outs)
print(is_valid("0001110100", 3, 2))

00 -> [('000', '00'), ('001', '01')]
01 -> [('010', '10'), ('011', '11')]
10 -> [('100', '00'), ('101', '01')]
11 -> [('110', '10'), ('111', '11')]
True


## 7. Các cách tiếp theo

### 7.1. Cách 2 — Hierholzer đệ quy trên đồ thị de Bruijn

**Giải thích.** Bắt đầu ở đỉnh `'0' * (n-1)`. Hàm `dfs(node)`:

- Với mỗi chữ số `x` từ `0` đến `k-1`: cạnh `e = node + x`. Nếu `e` chưa dùng → đánh dấu, gọi `dfs(e[1:])`, **sau khi gọi xong** mới ghi `x` vào kết quả (post-order).

Kết quả là danh sách chữ số theo thứ tự **ngược** của chu trình. Vì đồ thị de Bruijn đối xứng (đảo ngược mọi cạnh vẫn là đồ thị de Bruijn), nối thẳng không cần đảo cũng được — nhưng để dễ hiểu ta **đảo lại** cho đúng chiều, rồi đặt `start` ở đầu.

> Ghi chú: nhiều lời giải mẫu trả `''.join(ans) + start` (không đảo, đặt start ở cuối). Cả hai đều đúng; chiều nào cũng là một chu trình Euler hợp lệ.

In [10]:
def crack_hierholzer_rec(n, k):
    sys.setrecursionlimit(max(10000, 2 * k ** n + 100))
    start = '0' * (n - 1)
    used = set()
    digits = []

    def dfs(node):
        for x in map(str, range(k)):
            e = node + x
            if e not in used:
                used.add(e)
                dfs(e[1:])
                digits.append(x)     # post-order

    dfs(start)
    return start + ''.join(reversed(digits))

for n, k in [(1, 2), (2, 2), (3, 2), (2, 3), (4, 10)]:
    s = crack_hierholzer_rec(n, k)
    print(n, k, s if len(s) <= 20 else s[:20] + "...", is_valid(s, n, k))

1 2 01 True
2 2 00110 True
3 2 0001011100 True
2 3 0010211220 True
4 10 00001000200030004000... True


**Code nộp LeetCode (Cách 2).**

In [11]:
class SolutionRec:
    def crackSafe(self, n: int, k: int) -> str:
        import sys
        sys.setrecursionlimit(10000)
        start = '0' * (n - 1)
        used = set()
        ans = []

        def dfs(node):
            for x in map(str, range(k)):
                e = node + x
                if e not in used:
                    used.add(e)
                    dfs(e[1:])
                    ans.append(x)

        dfs(start)
        return ''.join(ans) + start   # chiều ngược cũng là chu trình Euler

print([is_valid(SolutionRec().crackSafe(n, k), n, k) for n, k in [(1, 1), (1, 2), (2, 2), (3, 3), (4, 8)]])

[True, True, True, True, True]


**Độ phức tạp.** Mỗi cạnh được xét một lần khi dùng; mỗi đỉnh thử `k` chữ số nhiều lần (mỗi lần quay lại đỉnh). Tổng `O(k^n · n)` (nhân `n` vì tạo chuỗi). Bộ nhớ `O(n · k^n)` cho `used`, độ sâu đệ quy `O(k^n)`.

### 7.2. Cách 3 — Hierholzer lặp (không đệ quy) với con trỏ cạnh

**Giải thích.** Thay đệ quy bằng stack tường minh. Với mỗi đỉnh, giữ **con trỏ** `ptr[u]` = chữ số tiếp theo cần thử (vì cạnh `u + x` chỉ dùng một lần, và ta thử `x` tăng dần, nên không cần tập `used` — con trỏ đủ để biết cạnh nào đã dùng!).

- Stack chứa các cặp `(đỉnh, chữ số đã gõ để tới đỉnh này)`.
- Đỉnh đỉnh-stack còn cạnh (`ptr[u] < k`): lấy `x = ptr[u]`, tăng `ptr[u]`, đẩy `(u[1:] + x, x)` (với `n=1` thì đỉnh luôn là `''`).
- Hết cạnh: pop, ghi chữ số vào kết quả.

Không cần `used` vì mỗi cạnh `u + x` chỉ đi ra từ đúng đỉnh `u` và con trỏ của `u` chỉ tăng.

In [12]:
def crack_hierholzer_iter(n, k):
    start = '0' * (n - 1)
    ptr = defaultdict(int)          # ptr[u] = chữ số tiếp theo chưa thử từ u
    stack = [(start, '')]
    out = []
    while stack:
        u, _ = stack[-1]
        if ptr[u] < k:
            x = str(ptr[u])
            ptr[u] += 1
            v = (u + x)[1:]
            stack.append((v, x))
        else:
            _, d = stack.pop()
            out.append(d)
    return start + ''.join(reversed(out))

for n, k in [(1, 1), (1, 3), (2, 2), (3, 2), (2, 10), (3, 10), (4, 8), (4, 10)]:
    if k ** n > 4096:
        continue
    s = crack_hierholzer_iter(n, k)
    print(n, k, len(s), is_valid(s, n, k))

1 1 1 True
1 3 3 True
2 2 5 True
3 2 10 True
2 10 101 True
3 10 1002 True
4 8 4099 True


Lưu ý phần tử `''` của đáy stack được pop cuối cùng, nên nằm ở đầu `reversed(out)` và không đóng góp ký tự nào — rất gọn.

> Cẩn thận: với `k = 10` chữ số `x` là một ký tự `'0'..'9'`; với `k > 10` (không có trong đề) cách biểu diễn bằng chuỗi sẽ sai vì `'10'` có 2 ký tự.

**Code nộp LeetCode (Cách 3).**

In [13]:
class SolutionIter:
    def crackSafe(self, n: int, k: int) -> str:
        start = '0' * (n - 1)
        ptr = {}
        stack = [(start, '')]
        out = []
        while stack:
            u = stack[-1][0]
            p = ptr.get(u, 0)
            if p < k:
                ptr[u] = p + 1
                x = str(p)
                stack.append(((u + x)[1:], x))
            else:
                out.append(stack.pop()[1])
        return start + ''.join(reversed(out))

print([is_valid(SolutionIter().crackSafe(n, k), n, k) for n, k in [(1, 1), (1, 2), (2, 2), (3, 3), (4, 8)]])

[True, True, True, True, True]


**Độ phức tạp.** Mỗi cạnh được đẩy và pop đúng một lần: `O(k^n)` thao tác stack, mỗi thao tác tạo chuỗi dài `n` → `O(n · k^n)`. Bộ nhớ `O(n · k^(n-1))` cho `ptr` + `O(n · k^n)` cho stack.

### 7.3. Cách 4 — Tham lam "thử chữ số lớn nhất trước"

**Giải thích.** Bắt đầu với `'0' * n`. Lặp: thử thêm chữ số `d` từ `k-1` **xuống** `0`; chọn `d` đầu tiên mà cửa sổ mới `s[-(n-1):] + d` chưa xuất hiện. Dừng khi không thêm được.

Kết quả nổi tiếng (thuật toán "prefer largest", tương tự "prefer one" của Martin 1934 cho nhị phân): tham lam này **không bao giờ kẹt sớm** và tạo ra chuỗi de Bruijn đầy đủ. Lý do sâu: bắt đầu từ `0...0`, chỉ có thể bị kẹt ở đỉnh `0^(n-1)` sau khi đã dùng hết cạnh vào của nó, và thứ tự "lớn trước" đảm bảo cạnh `0^n` (cạnh nhỏ nhất) được để dành cuối cùng ở mọi đỉnh, tạo thành một cây khung ngược hướng về `0^(n-1)` — đúng điều kiện của định lý BEST.

> Nếu thử **nhỏ trước** (`0` đến `k-1`) thì tham lam **có thể kẹt** — ta sẽ thấy ở mục 13.

In [14]:
def crack_greedy(n, k):
    s = '0' * n
    seen = {s}
    while True:
        tail = s[len(s) - n + 1:] if n > 1 else ''
        for d in range(k - 1, -1, -1):
            w = tail + str(d)
            if w not in seen:
                seen.add(w)
                s += str(d)
                break
        else:
            return s

ok = all(is_valid(crack_greedy(n, k), n, k)
         for n in range(1, 5) for k in range(1, 11) if k ** n <= 4096)
print("tham lam đúng với mọi (n, k) trong ràng buộc:", ok)
print(crack_greedy(2, 2), crack_greedy(3, 2), crack_greedy(2, 3))

tham lam đúng với mọi (n, k) trong ràng buộc: True
00110 0001110100 0022120110


**Độ phức tạp.** `k^n` lần thêm, mỗi lần thử tối đa `k` chữ số → `O(n · k^(n+1))`. Chú ý `s += str(d)` trên chuỗi Python có thể tốn `O(len)` — nên dùng list nếu cần nhanh. Với `k^n ≤ 4096` hoàn toàn ổn.

**Code nộp LeetCode (Cách 4).**

In [15]:
class SolutionGreedy:
    def crackSafe(self, n: int, k: int) -> str:
        res = ['0'] * n
        seen = {'0' * n}
        for _ in range(k ** n - 1):
            tail = ''.join(res[len(res) - n + 1:]) if n > 1 else ''
            for d in range(k - 1, -1, -1):
                w = tail + str(d)
                if w not in seen:
                    seen.add(w)
                    res.append(str(d))
                    break
        return ''.join(res)

print([is_valid(SolutionGreedy().crackSafe(n, k), n, k) for n, k in [(1, 1), (1, 2), (2, 2), (3, 3), (4, 8)]])

[True, True, True, True, True]


### 7.4. Cách 5 — Chuỗi de Bruijn bằng từ Lyndon (thuật toán FKM)

**Giải thích (mức giới thiệu).** Một **từ Lyndon** là chuỗi **nhỏ hơn hẳn** mọi phép xoay vòng của nó (ví dụ `001`, `011` là Lyndon; `010` không vì xoay thành `001` nhỏ hơn). Định lý Fredricksen–Kessler–Maiorana: nối **theo thứ tự từ điển** tất cả từ Lyndon có độ dài **chia hết `n`** sẽ được một **chuỗi de Bruijn vòng** độ dài `k^n`.

Ví dụ `n = 3, k = 2`: Lyndon có độ dài ∈ {1, 3}: `0, 001, 011, 1` → `0 001 011 1` = `00010111` (vòng). Để thành chuỗi thẳng, nối thêm `n-1` ký tự đầu: `0001011100`.

Thuật toán đệ quy cổ điển `db(t, p)` sinh các từ này không cần kiểm tra từng chuỗi. Đây là cách "toán học" nhất, chạy `O(k^n)`.

In [16]:
def de_bruijn_fkm(k, n):
    a = [0] * (k * n + 1)
    seq = []

    def db(t, p):
        if t > n:
            if n % p == 0:
                seq.extend(a[1:p + 1])
        else:
            a[t] = a[t - p]
            db(t + 1, p)
            for j in range(a[t - p] + 1, k):
                a[t] = j
                db(t + 1, t)

    db(1, 1)
    return seq

def crack_lyndon(n, k):
    seq = de_bruijn_fkm(k, n)
    m = len(seq)
    lin = seq + [seq[i % m] for i in range(n - 1)]   # nối thêm n-1 ký tự (quay vòng)
    return ''.join(map(str, lin))

print(de_bruijn_fkm(2, 3), crack_lyndon(3, 2))
ok = all(is_valid(crack_lyndon(n, k), n, k)
         for n in range(1, 5) for k in range(1, 11) if k ** n <= 4096)
print("FKM đúng với mọi (n, k):", ok)

[0, 0, 0, 1, 0, 1, 1, 1] 0001011100
FKM đúng với mọi (n, k): True


**Code nộp LeetCode (Cách 5).**

In [17]:
class SolutionLyndon:
    def crackSafe(self, n: int, k: int) -> str:
        a = [0] * (k * n + 1)
        seq = []

        def db(t, p):
            if t > n:
                if n % p == 0:
                    seq.extend(a[1:p + 1])
            else:
                a[t] = a[t - p]
                db(t + 1, p)
                for j in range(a[t - p] + 1, k):
                    a[t] = j
                    db(t + 1, t)

        db(1, 1)
        m = len(seq)
        seq += [seq[i % m] for i in range(n - 1)]
        return ''.join(map(str, seq))

print([is_valid(SolutionLyndon().crackSafe(n, k), n, k) for n, k in [(1, 1), (1, 2), (2, 2), (3, 3), (4, 8)]])

[True, True, True, True, True]


**Độ phức tạp.** Độ sâu đệ quy chỉ `n + 1` (không phải `k^n`!), tổng thời gian `O(k^n)` khấu hao. Bộ nhớ `O(k^n)` cho kết quả.

## 8. TRACE từng bước

Ta trace **Cách 3 (Hierholzer lặp)** với `n = 2, k = 2`. Đỉnh là chuỗi độ dài 1: `'0'`, `'1'`. Bắt đầu `start = '0'`.

| bước | đỉnh đỉnh-stack | ptr trước | hành động | stack (đỉnh, chữ số) | out |
|---|---|---|---|---|---|
| 1 | `0` | ptr[0]=0 | đi cạnh `00` → đỉnh `0` | `(0,''),(0,'0')` | |
| 2 | `0` | ptr[0]=1 | đi cạnh `01` → đỉnh `1` | `...,(1,'1')` | |
| 3 | `1` | ptr[1]=0 | đi cạnh `10` → đỉnh `0` | `...,(0,'0')` | |
| 4 | `0` | ptr[0]=2 | hết cạnh, pop | | `0` |
| 5 | `1` | ptr[1]=1 | đi cạnh `11` → đỉnh `1` | `...,(1,'1')` | |
| 6–9 | | | lần lượt pop hết | | `0,1,1,0,''` |

`out` đảo ngược: `'' 0 1 1 0` → nối sau `start='0'` → `"00110"`. Đối chiếu bằng hàm trace bên dưới.

In [18]:
def trace_hierholzer(n, k):
    start = '0' * (n - 1)
    ptr = defaultdict(int)
    stack = [(start, '')]
    out = []
    step = 0
    while stack:
        step += 1
        u, _ = stack[-1]
        if ptr[u] < k:
            x = str(ptr[u])
            ptr[u] += 1
            v = (u + x)[1:]
            stack.append((v, x))
            print(f"b{step:2}: tại '{u}' đi cạnh '{u + x}' -> '{v}'   stack={[d for _, d in stack]}")
        else:
            _, d = stack.pop()
            out.append(d)
            print(f"b{step:2}: '{u}' hết cạnh, pop chữ số {d!r:4}   out={out}")
    res = start + ''.join(reversed(out))
    print("kết quả:", res, "hợp lệ:", is_valid(res, n, k))
    return res

trace_hierholzer(2, 2)

b 1: tại '0' đi cạnh '00' -> '0'   stack=['', '0']
b 2: tại '0' đi cạnh '01' -> '1'   stack=['', '0', '1']
b 3: tại '1' đi cạnh '10' -> '0'   stack=['', '0', '1', '0']
b 4: '0' hết cạnh, pop chữ số '0'    out=['0']
b 5: tại '1' đi cạnh '11' -> '1'   stack=['', '0', '1', '1']
b 6: '1' hết cạnh, pop chữ số '1'    out=['0', '1']
b 7: '1' hết cạnh, pop chữ số '1'    out=['0', '1', '1']
b 8: '0' hết cạnh, pop chữ số '0'    out=['0', '1', '1', '0']
b 9: '0' hết cạnh, pop chữ số ''     out=['0', '1', '1', '0', '']
kết quả: 00110 hợp lệ: True


'00110'

Trace lớn hơn một chút: `n = 3, k = 2` (8 cạnh). Quan sát lúc đỉnh `00` "hết cạnh" sớm và các vòng phụ được ghép vào.

In [19]:
trace_hierholzer(3, 2)

b 1: tại '00' đi cạnh '000' -> '00'   stack=['', '0']
b 2: tại '00' đi cạnh '001' -> '01'   stack=['', '0', '1']
b 3: tại '01' đi cạnh '010' -> '10'   stack=['', '0', '1', '0']
b 4: tại '10' đi cạnh '100' -> '00'   stack=['', '0', '1', '0', '0']
b 5: '00' hết cạnh, pop chữ số '0'    out=['0']
b 6: tại '10' đi cạnh '101' -> '01'   stack=['', '0', '1', '0', '1']
b 7: tại '01' đi cạnh '011' -> '11'   stack=['', '0', '1', '0', '1', '1']
b 8: tại '11' đi cạnh '110' -> '10'   stack=['', '0', '1', '0', '1', '1', '0']
b 9: '10' hết cạnh, pop chữ số '0'    out=['0', '0']
b10: tại '11' đi cạnh '111' -> '11'   stack=['', '0', '1', '0', '1', '1', '1']
b11: '11' hết cạnh, pop chữ số '1'    out=['0', '0', '1']
b12: '11' hết cạnh, pop chữ số '1'    out=['0', '0', '1', '1']
b13: '01' hết cạnh, pop chữ số '1'    out=['0', '0', '1', '1', '1']
b14: '10' hết cạnh, pop chữ số '0'    out=['0', '0', '1', '1', '1', '0']
b15: '01' hết cạnh, pop chữ số '1'    out=['0', '0', '1', '1', '1', '0', '1']
b16: '00' hế

'0001011100'

Trace **tham lam** (Cách 4) với `n = 2, k = 3`: in cửa sổ được thêm ở mỗi bước.

In [20]:
def trace_greedy(n, k):
    s = '0' * n
    seen = {s}
    print("bắt đầu:", s)
    while True:
        tail = s[len(s) - n + 1:] if n > 1 else ''
        for d in range(k - 1, -1, -1):
            w = tail + str(d)
            if w not in seen:
                seen.add(w)
                s += str(d)
                print(f"  tail='{tail}' thử từ {k-1} xuống, chọn {d} -> cửa sổ '{w}', s = {s}")
                break
        else:
            print("  không thêm được nữa. Kết quả:", s, is_valid(s, n, k))
            return s

trace_greedy(2, 3)

bắt đầu: 00
  tail='0' thử từ 2 xuống, chọn 2 -> cửa sổ '02', s = 002
  tail='2' thử từ 2 xuống, chọn 2 -> cửa sổ '22', s = 0022
  tail='2' thử từ 2 xuống, chọn 1 -> cửa sổ '21', s = 00221
  tail='1' thử từ 2 xuống, chọn 2 -> cửa sổ '12', s = 002212
  tail='2' thử từ 2 xuống, chọn 0 -> cửa sổ '20', s = 0022120
  tail='0' thử từ 2 xuống, chọn 1 -> cửa sổ '01', s = 00221201
  tail='1' thử từ 2 xuống, chọn 1 -> cửa sổ '11', s = 002212011
  tail='1' thử từ 2 xuống, chọn 0 -> cửa sổ '10', s = 0022120110
  không thêm được nữa. Kết quả: 0022120110 True


'0022120110'

## 9. Code nộp LeetCode

Bản chọn để nộp: **Hierholzer lặp** (Cách 3) — không lo giới hạn đệ quy, tuyến tính theo số cạnh, có chứng minh rõ ràng.

In [21]:
class Solution:
    def crackSafe(self, n: int, k: int) -> str:
        start = '0' * (n - 1)          # đỉnh xuất phát (độ dài n-1)
        ptr = {}                       # ptr[u]: chữ số tiếp theo chưa thử từ đỉnh u
        stack = [(start, '')]          # (đỉnh, chữ số đã gõ để tới đỉnh)
        out = []
        while stack:
            u = stack[-1][0]
            p = ptr.get(u, 0)
            if p < k:                  # còn cạnh u + str(p) chưa dùng
                ptr[u] = p + 1
                x = str(p)
                stack.append(((u + x)[1:], x))
            else:                      # hết cạnh: ghi post-order
                out.append(stack.pop()[1])
        return start + ''.join(reversed(out))

sol = Solution()
print(sol.crackSafe(1, 2), sol.crackSafe(2, 2))
assert is_valid(sol.crackSafe(1, 2), 1, 2)
assert is_valid(sol.crackSafe(2, 2), 2, 2)

01 00110


Giải thích từng dòng:

- `start = '0' * (n - 1)`: đỉnh xuất phát. Với `n = 1`, đây là chuỗi rỗng — đồ thị chỉ có một đỉnh `''` với `k` khuyên.
- `ptr`: thay cho tập `used`. Cạnh rời `u` được thử theo thứ tự `0, 1, ..., k-1`; khi `ptr[u] == k` nghĩa là mọi cạnh ra khỏi `u` đã dùng.
- `stack.append(((u + x)[1:], x))`: đi theo cạnh `u + x`, đỉnh mới là cạnh bỏ ký tự đầu.
- `out.append(stack.pop()[1])`: post-order, ghi chữ số đã dùng để tới đỉnh vừa "chết".
- `start + reversed(out)`: `n - 1` chữ số đầu là đỉnh xuất phát, sau đó `k^n` chữ số ứng với `k^n` cạnh. Phần tử đáy stack `(start, '')` bị pop **cuối cùng**, nên `''` nằm ở **đầu** của `reversed(out)` và không thêm ký tự nào.

## 10. Vì sao thuật toán đúng

1. **Cận dưới**: mọi đáp án dài ít nhất `k^n + n - 1` (mục 4.1).
2. **Tương ứng**: một chuỗi `s` độ dài `k^n + n - 1` chứa mọi mật khẩu ⇔ dãy cửa sổ `s[i:i+n]` là một **đường đi Euler** trong đồ thị de Bruijn `B(k, n-1)`: cửa sổ thứ `i` là cạnh từ `s[i:i+n-1]` tới `s[i+1:i+n]`, và hai cửa sổ liên tiếp có chung đỉnh.
3. **Tồn tại**: mọi đỉnh có `in = out = k` và đồ thị liên thông mạnh ⇒ có chu trình Euler.
4. **Hierholzer tìm đúng chu trình Euler**: bất biến — dãy đỉnh trên stack luôn là một đường đi bằng các cạnh đã dùng; một đỉnh chỉ bị pop khi mọi cạnh ra đã dùng; dãy post-order đảo ngược là đường đi dùng mọi cạnh đúng một lần (các vòng phụ được "chèn" vào đúng chỗ đỉnh mà chúng xuất phát).
5. Vì mỗi cạnh được đẩy lên stack đúng một lần, `out` có đúng `k^n + 1` phần tử (gồm `''` của đáy) → độ dài kết quả `(n - 1) + k^n`. Đạt cận dưới ⇒ tối ưu.

**Tham lam (Cách 4)** đúng nhờ lập luận cây khung (định lý BEST): cạnh `0` luôn là cạnh dùng cuối cùng ở mỗi đỉnh khác `0^(n-1)`, và các cạnh "cuối cùng" này tạo thành cây hướng về `0^(n-1)`, nên quá trình chỉ dừng khi đã dùng hết cạnh. **FKM (Cách 5)** đúng theo định lý Fredricksen–Maiorana.

## 11. Phân tích Time/Space

Gọi `E = k^n` (số cạnh = số mật khẩu), `V = k^(n-1)`.

- **Backtracking**: không quay lui thì `O(n · k · E)`; tệ nhất lý thuyết mũ. Space `O(n · E)`.
- **Hierholzer đệ quy**: `O(n · k · E)` (mỗi lần quay về đỉnh lại quét `k` chữ số và tra `used`). Space `O(n · E)` + stack đệ quy `O(E)`.
- **Hierholzer lặp + con trỏ**: `O(n · E)`. Space `O(n · E)`.
- **Tham lam**: `O(n · k · E)`. Space `O(n · E)`.
- **FKM**: `O(E)` khấu hao, độ sâu đệ quy `O(n)`. Space `O(E)`.

Với `E ≤ 4096`, mọi cách đều chạy tức thì; chỉ backtracking là có rủi ro lý thuyết.

In [22]:
import time
for name, f in [("backtrack", lambda n, k: crack_backtrack(n, k)[0]),
                ("hier_rec", crack_hierholzer_rec), ("hier_iter", crack_hierholzer_iter),
                ("greedy", crack_greedy), ("lyndon", crack_lyndon)]:
    t = time.perf_counter()
    for _ in range(5):
        s = f(4, 8)        # 8^4 = 4096 mật khẩu
    print(f"{name:10} {1000 * (time.perf_counter() - t) / 5:7.2f} ms  valid={is_valid(s, 4, 8)}")

backtrack     6.00 ms  valid=True


hier_rec      7.01 ms  valid=True
hier_iter     2.05 ms  valid=True


greedy        3.63 ms  valid=True
lyndon        0.41 ms  valid=True


## 12. Bộ test

Vì có nhiều đáp án, mọi test đều dùng `is_valid`. Ta kiểm tra **mọi** cặp `(n, k)` hợp lệ theo ràng buộc cho cả 5 cách và các lớp `Solution`.

In [23]:
# 12.1 Ví dụ đề
assert is_valid(Solution().crackSafe(1, 2), 1, 2)
assert is_valid(Solution().crackSafe(2, 2), 2, 2)

# 12.2 Edge cases
assert Solution().crackSafe(1, 1) == "0"
assert is_valid(Solution().crackSafe(4, 1), 4, 1) and Solution().crackSafe(4, 1) == "0000"
assert sorted(Solution().crackSafe(1, 10)) == list("0123456789")
assert is_valid(Solution().crackSafe(4, 8), 4, 8)    # k^n = 4096 (lớn nhất)
assert is_valid(Solution().crackSafe(2, 10), 2, 10)

# 12.3 Toàn bộ không gian ràng buộc, mọi cách
cases = [(n, k) for n in range(1, 5) for k in range(1, 11) if k ** n <= 4096]
methods = {
    "backtrack": lambda n, k: crack_backtrack(n, k)[0],
    "hier_rec": crack_hierholzer_rec,
    "hier_iter": crack_hierholzer_iter,
    "greedy": crack_greedy,
    "lyndon": crack_lyndon,
    "SolutionRec": lambda n, k: SolutionRec().crackSafe(n, k),
    "SolutionIter": lambda n, k: SolutionIter().crackSafe(n, k),
    "SolutionGreedy": lambda n, k: SolutionGreedy().crackSafe(n, k),
    "SolutionLyndon": lambda n, k: SolutionLyndon().crackSafe(n, k),
    "Solution": lambda n, k: Solution().crackSafe(n, k),
}
for name, f in methods.items():
    for n, k in cases:
        assert is_valid(f(n, k), n, k), (name, n, k)
print("Tất cả", len(cases), "cặp (n, k) x", len(methods), "cách: OK")

Tất cả 38 cặp (n, k) x 10 cách: OK


In [24]:
# 12.4 Kiểm tra chính hàm is_valid bằng brute force trên chuỗi ngẫu nhiên (có seed)
import random
random.seed(753)

def is_valid_brute(s, n, k):
    if len(s) != k ** n + n - 1:
        return False
    for p in product(map(str, range(k)), repeat=n):
        if ''.join(p) not in s:
            return False
    return all(c in '0123456789'[:k] for c in s)

cnt_true = 0
for _ in range(3000):
    n = random.randint(1, 3)
    k = random.randint(1, 3)
    L = k ** n + n - 1
    s = ''.join(random.choice('0123456789'[:k]) for _ in range(L))
    a, b = is_valid(s, n, k), is_valid_brute(s, n, k)
    assert a == b, (s, n, k)
    cnt_true += a
print("is_valid khớp brute force; số chuỗi ngẫu nhiên hợp lệ:", cnt_true)

is_valid khớp brute force; số chuỗi ngẫu nhiên hợp lệ: 1305


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Tham lam "nhỏ trước" (hoặc DFS không quay lui, ghi pre-order)

**Code sai:**

In [25]:
def wrong_greedy_small_first(n, k):
    s = '0' * n
    seen = {s}
    while True:
        tail = s[len(s) - n + 1:] if n > 1 else ''
        for d in range(k):                 # SAI: thử 0 trước
            w = tail + str(d)
            if w not in seen:
                seen.add(w); s += str(d); break
        else:
            return s

bad = [(n, k) for n, k in cases if not is_valid(wrong_greedy_small_first(n, k), n, k)]
print("số (n, k) bị sai:", len(bad), "ví dụ:", bad[:5])
print(wrong_greedy_small_first(2, 2))   # "00" + thử 0 -> "00" đã có, thử 1 -> ...

số (n, k) bị sai: 25 ví dụ: [(2, 2), (2, 3), (2, 4), (2, 5), (2, 6)]
0010


**Vì sao sai:** ví dụ `n = 2, k = 2`: `00 → 001 → 0010`, lúc này đuôi là `0`, cả `00` và `01` đều đã dùng → kẹt khi còn thiếu `11`. Tổng quát, thứ tự nhỏ-trước dễ đưa ta quay lại đỉnh `0^(n-1)` khi nó đã hết cạnh ra, trong khi các đỉnh khác vẫn còn cạnh chưa dùng → kẹt sớm, chuỗi ngắn hơn yêu cầu.

**Code đúng:** thử **lớn trước** (`range(k-1, -1, -1)`) như Cách 4, hoặc dùng Hierholzer với **post-order**.

### Lỗi 2 — Ghi chữ số theo pre-order trong Hierholzer

**Code sai:**

In [26]:
def wrong_preorder(n, k):
    sys.setrecursionlimit(20000)
    start = '0' * (n - 1)
    used, ans = set(), []

    def dfs(node):
        for x in map(str, range(k)):
            e = node + x
            if e not in used:
                used.add(e)
                ans.append(x)        # SAI: ghi trước khi đệ quy
                dfs(e[1:])

    dfs(start)
    return start + ''.join(ans)

bad = [(n, k) for n, k in cases if not is_valid(wrong_preorder(n, k), n, k)]
print("pre-order sai ở", len(bad), "trường hợp, ví dụ:", bad[:4])

pre-order sai ở 25 trường hợp, ví dụ: [(2, 2), (2, 3), (2, 4), (2, 5)]


**Vì sao sai:** pre-order ghi các cạnh theo thứ tự khám phá; khi DFS quay lui và đi một "vòng phụ" từ một đỉnh giữa đường, vòng phụ bị **nối vào cuối** chuỗi thay vì **chèn vào giữa**, làm đứt tính liên tiếp của các cửa sổ.

**Code đúng:** `dfs(e[1:])` **trước**, `ans.append(x)` **sau** (post-order) — như Cách 2.

### Lỗi 3 — Dùng đỉnh là chuỗi độ dài `n` thay vì `n - 1` (rồi quên cửa sổ đầu)

**Code sai:**

In [27]:
def wrong_offset(n, k):
    # Hierholzer nhưng start dài n, và mỗi bước lấy cạnh độ dài n+1
    start = '0' * n
    ptr = defaultdict(int)
    stack, out = [(start, '')], []
    while stack:
        u, _ = stack[-1]
        if ptr[u] < k:
            x = str(ptr[u]); ptr[u] += 1
            stack.append(((u + x)[1:], x))
        else:
            out.append(stack.pop()[1])
    return start + ''.join(reversed(out))

s = wrong_offset(2, 2)
print(s, len(s), "cần", 2 ** 2 + 2 - 1, "->", is_valid(s, 2, 2))

0001011100 10 cần 5 -> False


**Vì sao sai:** đỉnh dài `n` nghĩa là cạnh dài `n + 1` — ta đang dựng chuỗi de Bruijn bậc `n + 1`, dài `k^(n+1) + n`, quá dài. Vẫn chứa mọi mật khẩu nhưng **không tối ưu**.

**Code đúng:** đỉnh là chuỗi **`n - 1`** ký tự, `start = '0' * (n - 1)`.

### Lỗi 4 — Quên tăng giới hạn đệ quy

Với `n = 4, k = 8`, Hierholzer đệ quy sâu ~4096 tầng > 1000 mặc định → `RecursionError`. Sửa: `sys.setrecursionlimit(10000)` hoặc dùng bản lặp.

In [28]:
def demo_recursion_limit():
    old = sys.getrecursionlimit()
    sys.setrecursionlimit(1000)
    try:
        start, used, ans = '000', set(), []
        def dfs(node):
            for x in map(str, range(8)):
                e = node + x
                if e not in used:
                    used.add(e); dfs(e[1:]); ans.append(x)
        dfs(start)
        print("không lỗi?!")
    except RecursionError:
        print("RecursionError như dự đoán")
    finally:
        sys.setrecursionlimit(max(old, 20000))

demo_recursion_limit()

RecursionError như dự đoán


## 14. Mẹo & nhận dạng pattern

- **"Chuỗi ngắn nhất chứa mọi chuỗi con độ dài n"** → chuỗi **de Bruijn**, độ dài `k^n + n - 1` (thẳng) hoặc `k^n` (vòng).
- **"Đi qua mọi cạnh đúng một lần"** → Euler / Hierholzer (post-order, đảo ngược). Các bài cùng họ: *332. Reconstruct Itinerary*, *2097. Valid Arrangement of Pairs*.
- **Đổi Hamilton thành Euler**: khi đỉnh là các chuỗi chồng lấn `n` ký tự, xem chúng là **cạnh** giữa các chuỗi `n-1` ký tự (giống ý tưởng lắp ráp DNA trong tin sinh học).
- **Con trỏ cạnh thay tập `used`**: nếu cạnh ra của mỗi đỉnh có thể đánh số `0..deg-1`, chỉ cần một số nguyên mỗi đỉnh.
- **Bài có nhiều đáp án** → viết hàm `is_valid` thay vì so sánh chuỗi.

## 15. Biến thể

1. **Chuỗi de Bruijn vòng** (két dạng vòng tròn): bỏ `n - 1` ký tự lặp cuối — dài đúng `k^n`.
2. **Bảng chữ cái tuỳ ý** (ví dụ chữ cái `'a'..'z'`): thay `str(x)` bằng `alphabet[x]`; với `k > 10` không dùng được chữ số đơn.
3. **Chỉ cần phủ một tập con mật khẩu** → không còn là Euler, trở thành bài toán NP-khó (shortest common superstring).
4. **Đường Euler tổng quát** (đỉnh không cân bằng): bắt đầu ở đỉnh có `out - in = 1` — xem *332*, *2097*.
5. **Đếm số chuỗi de Bruijn**: `(k!)^(k^(n-1)) / k^n` (theo định lý BEST) — ví dụ `n = 2, k = 2` có `2^2/4 = 1` chuỗi vòng.

In [29]:
# Biến thể 1: chuỗi de Bruijn vòng
def de_bruijn_cyclic(n, k):
    s = Solution().crackSafe(n, k)
    return s[:k ** n]

def is_cyclic_valid(c, n, k):
    ext = c + c[:n - 1]
    return len(c) == k ** n and len({ext[i:i + n] for i in range(len(c))}) == k ** n

print(de_bruijn_cyclic(3, 2), is_cyclic_valid(de_bruijn_cyclic(3, 2), 3, 2))

# Biến thể 2: bảng chữ tuỳ ý
def de_bruijn_alphabet(n, alphabet):
    k = len(alphabet)
    digits = Solution().crackSafe(n, k) if k <= 10 else None
    return ''.join(alphabet[int(c)] for c in digits)

print(de_bruijn_alphabet(2, "ACGT"))

00010111 True
AACAGATCCGCTGGTTA


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Backtracking (Hamilton) | DFS qua các mật khẩu, quay lui khi kẹt | `O(n·k·k^n)` thực tế, mũ lý thuyết | `O(n·k^n)` | Phỏng vấn: khởi đầu dễ nghĩ |
| 2. Hierholzer đệ quy | Euler trên đồ thị `n-1`, post-order | `O(n·k·k^n)` | `O(n·k^n)` + đệ quy sâu | Lời giải ngắn gọn nhất |
| 3. Hierholzer lặp + con trỏ | Như 2, stack tự quản | `O(n·k^n)` | `O(n·k^n)` | An toàn nhất, nên nộp |
| 4. Tham lam lớn-trước | Luôn thử chữ số lớn nhất chưa tạo cửa sổ cũ | `O(n·k^(n+1))` | `O(n·k^n)` | Code cực ngắn, cần nhớ "lớn trước" |
| 5. FKM / Lyndon | Nối từ Lyndon có độ dài chia hết n | `O(k^n)` | `O(k^n)` | Khi cần tốc độ / sinh de Bruijn lớn |

## 17. Bài tập

**Bài 1 (đoán output).** `crack_greedy(2, 3)` trả về chuỗi gì? `crack_lyndon(2, 3)`?

**Bài 2 (tính tay).** Liệt kê các từ Lyndon trên `{0,1}` có độ dài chia hết 4 theo thứ tự từ điển, rồi viết chuỗi de Bruijn vòng `B(2, 4)`.

**Bài 3 (điền TODO).** Hoàn thiện `count_windows(s, n)` trả về số cửa sổ độ dài `n` **khác nhau** của `s`.

**Bài 4 (tự viết lại).** Viết lại Hierholzer **đệ quy** nhưng dùng con trỏ `ptr` thay cho tập `used`.

**Bài 5 (mini challenge).** Viết `crack_safe_alpha(n, alphabet)` hoạt động với **bảng chữ bất kỳ** (kể cả `k > 10`), trả về **danh sách ký hiệu** độ dài `k^n + n - 1`, và hàm kiểm tra tương ứng.

In [30]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def count_windows(s, n):
    # TODO: dùng set comprehension trên s[i:i+n]
    pass

# Bài 4 — TODO
def crack_rec_ptr(n, k):
    # TODO: ptr = defaultdict(int); dfs(u): while ptr[u] < k: x = ptr[u]; ptr[u] += 1; dfs(...); ans.append(...)
    pass

# Bài 5 — TODO
def crack_safe_alpha(n, alphabet):
    # TODO: làm việc với chỉ số 0..k-1 dưới dạng tuple thay vì chuỗi
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: chạy tay theo bảng ở mục 8 (trace_greedy).
- Mức 2: từ `00`, thử `2` trước → `02`, rồi `22`, ...
- Mức 3: tham lam cho `"0022120110"`; FKM: Lyndon `0, 01, 02, 1, 12, 2` → vòng `001021122`, thẳng `0010211220`.

**Bài 2.**
- Mức 1: độ dài chia hết 4: 1, 2, 4.
- Mức 2: Lyndon độ dài 1: `0, 1`; độ dài 2: `01`; độ dài 4: `0001, 0011, 0111`.
- Mức 3: thứ tự từ điển: `0, 0001, 0011, 01, 0111, 1` → `0000100110101111`.

**Bài 3.**
- Mức 1: `len(s) - n + 1` cửa sổ.
- Mức 2: đưa vào `set`.
- Mức 3: `len({s[i:i+n] for i in range(len(s)-n+1)})`.

**Bài 4.**
- Mức 1: giữ `ptr[u]` như Cách 3.
- Mức 2: vòng `while ptr[u] < k` thay cho `for`.
- Mức 3: nhớ tăng recursion limit; ghép `start + reversed(ans)`.

**Bài 5.**
- Mức 1: biểu diễn đỉnh là tuple chỉ số.
- Mức 2: cạnh `u + (x,)`, đỉnh mới `(u + (x,))[1:]`.
- Mức 3: kiểm tra bằng tập các tuple cửa sổ.

## 19. Đáp án tham khảo

In [31]:
# Bài 1
assert crack_greedy(2, 3) == "0022120110", crack_greedy(2, 3)
assert crack_lyndon(2, 3) == "0010211220", crack_lyndon(2, 3)
assert is_valid("0022120110", 2, 3) and is_valid("0010211220", 2, 3)

# Bài 2
assert ''.join(map(str, de_bruijn_fkm(2, 4))) == "0000100110101111"

# Bài 3
def count_windows_ans(s, n):
    return len({s[i:i + n] for i in range(len(s) - n + 1)})
assert count_windows_ans("01100", 2) == 4
assert count_windows_ans("0000", 2) == 1

# Bài 4
def crack_rec_ptr_ans(n, k):
    sys.setrecursionlimit(20000)
    start = '0' * (n - 1)
    ptr = defaultdict(int)
    ans = []
    def dfs(u):
        while ptr[u] < k:
            x = str(ptr[u]); ptr[u] += 1
            dfs((u + x)[1:])
            ans.append(x)
    dfs(start)
    return start + ''.join(reversed(ans))
assert all(is_valid(crack_rec_ptr_ans(n, k), n, k) for n, k in cases)

# Bài 5
def crack_safe_alpha_ans(n, alphabet):
    k = len(alphabet)
    start = (0,) * (n - 1)
    ptr = defaultdict(int)
    stack, out = [(start, None)], []
    while stack:
        u = stack[-1][0]
        if ptr[u] < k:
            x = ptr[u]; ptr[u] += 1
            stack.append(((u + (x,))[1:], x))
        else:
            d = stack.pop()[1]
            if d is not None:
                out.append(d)
    seq = list(start) + out[::-1]
    return [alphabet[i] for i in seq]

def is_valid_alpha(seq, n, alphabet):
    k = len(alphabet)
    wins = {tuple(seq[i:i + n]) for i in range(len(seq) - n + 1)}
    return len(seq) == k ** n + n - 1 and len(wins) == k ** n

alpha12 = list("abcdefghijkl")             # k = 12 > 10
seq = crack_safe_alpha_ans(3, alpha12)
assert is_valid_alpha(seq, 3, alpha12)
assert is_valid_alpha(crack_safe_alpha_ans(2, list("ACGT")), 2, list("ACGT"))
print("Đáp án đều đúng. Ví dụ ACGT:", ''.join(crack_safe_alpha_ans(2, list("ACGT"))))

Đáp án đều đúng. Ví dụ ACGT: AACAGATCCGCTGGTTA


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template Hierholzer trên de Bruijn (lặp):**

```python
start = '0' * (n - 1)
ptr, stack, out = {}, [(start, '')], []
while stack:
    u = stack[-1][0]; p = ptr.get(u, 0)
    if p < k:
        ptr[u] = p + 1; x = str(p)
        stack.append(((u + x)[1:], x))   # đi cạnh u+x
    else:
        out.append(stack.pop()[1])       # post-order
return start + ''.join(reversed(out))
```

**Template đệ quy ngắn (nhớ setrecursionlimit):**

```python
def dfs(u):
    for x in map(str, range(k)):
        if u + x not in used:
            used.add(u + x); dfs((u + x)[1:]); ans.append(x)
dfs(start); return ''.join(ans) + start
```

**Checklist trước khi submit:**

- [ ] Đỉnh là chuỗi độ dài **`n - 1`**, không phải `n`.
- [ ] Ghi chữ số **post-order** (sau khi đệ quy / khi pop).
- [ ] Ghép thêm `start` (`n - 1` chữ số) vào kết quả.
- [ ] Độ dài kết quả đúng `k^n + n - 1`.
- [ ] `n = 1` chạy được (đỉnh rỗng `''`).
- [ ] Đệ quy sâu `k^n` → `setrecursionlimit` hoặc bản lặp.
- [ ] Tham lam thì thử chữ số **lớn trước**.
- [ ] Tên method đúng: `crackSafe(self, n, k)`.